### RAG 심화
- 중복 문서 문제 : 비슷한 내용의 chunk 여러개 반환되어 컨텍스트 낭비
- 검색 : 시멘틱 유사도만으로는 정확한 키워드 매칭 어려움
- 구조적 질의 불가 : ex) 2024년 이후 계약 금액이 1억 이상인 제품 찾기 = 메타필터 처리 불가
- 노이즈 청크 : 관련성이 낮은 청크가 LLM에게 전달되어 환각 유발

In [6]:
# 라이브러리 로드
from langchain_ollama import ChatOllama
from langchain_ibm import ChatWatsonx
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    PydanticOutputParser,
)
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableLambda,
    RunnableParallel,
)
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.chat_history import (
    InMemoryChatMessageHistory,
    BaseChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv
import os
import gradio as gr

# 모델(LLM, Embedding)
from langchain_community.document_loaders import (
    PyPDFLoader,
    CSVLoader,
    WebBaseLoader,
    DirectoryLoader,
)
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_ibm import WatsonxEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers import (
    EnsembleRetriever, 
    ContextualCompressionRetriever, 
    BM25Retriever
)
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

from langchain_cohere import CohereRerank

from langchain_classic.retrievers.document_compressors import LLMChainExtractor, EmbeddingsFilter, DocumentCompressorPipeline

import fitz
import easyocr
import json

In [2]:
#.env 내용 가죠오기
load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.getenv("HF_TOKEN")
cohere_api_key = os.getenv("COHERE_API_KEY")

In [3]:
from langchain_openai import ChatOpenAI

# HuggingFace model

hugging_llm = ChatOpenAI(
    model="Qwen/Qwen2.5-7B-Instruct:together",
    api_key=hf_token,
    base_url="https://router.huggingface.co/v1",
    temperature= 0
)
# 유료 LLM 선언

watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}",
    params = {
    "max_tokens": 2000,
    "temperature": 0
    }
)

# 로컬 LLM 선언
qwen_llm = ChatOllama(model="qwen3.5:4b",temperature= 0)

exaone_llm = ChatOllama(model="exaone3.5:2.4b",temperature= 0)

In [4]:
ollama_enbedding = OllamaEmbeddings(model="nomic-embed-text-v2-moe")

watsonx_enbedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}"
    )

In [5]:
# pdf => chunks 반환 함수
def create_chunk_from_pdf(pdf_path,chunk_size=500,chunk_overlap=50):
    loder = PyPDFLoader(pdf_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(loder.load())

    # 공백 제거
    chunks = [chunk for chunk in chunks if chunk.page_content.strip()]
    return chunks

def create_vectorstore(chunks, embeddings, collection_name,persist_directory='./db/chroma_db'):
    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name
    )

def create_retriever(vectorstore,search_type="similarity",k=3,fetch_k=20,lambda_mult=0.5):
    kwargs = {"k":k}

    if search_type=="mmr":
        kwargs['fetch_k']= fetch_k
        kwargs['lambda_mult'] = lambda_mult

    return vectorstore.as_retriever(search_type=search_type,search_kwargs=kwargs)

def print_retrieved_docs(title, retriever, query):
    docs = retriever.invoke(query)
    
    print("\n"+"="*50)
    print(title)
    print("="*50)

    for i, doc in enumerate(docs):
        print(f"\n[chunk {i}]")
        print(doc.page_content)
        print(f"\nPage : {doc.metadata.get("page")}")

### 1. 임베딩 코델, 청크 사이즈, 오버랩

In [ ]:
chunk1 = create_chunk_from_pdf("./data/Summary of ChatGPTGPT-4 Research.pdf",1000,100)
chunk2 = create_chunk_from_pdf("./data/Summary of ChatGPTGPT-4 Research.pdf",300,30)

print(f"분할된 청크 수 : {len(chunk1)}")
print(f"분할된 청크 수 : {len(chunk2)}")

분할된 청크 수 :118
분할된 청크 수 :378


In [20]:
vectorstore1 = create_vectorstore(chunk1,watsonx_enbedding,collection_name="gtp_research_watson1")
vectorstore2 = create_vectorstore(chunk2,watsonx_enbedding,collection_name="gtp_research_watson2")

watson1_retriever = create_retriever(vectorstore1)
watson2_retriever = create_retriever(vectorstore2)

query = 'where ci use chatGPT?'

print_retrieved_docs('chunk=1000,overlap=100',watson1_retriever, query)
print_retrieved_docs('chunk=300,overlap=30',watson2_retriever, query)


chunk=1000,overlap=100

[chunk 0]
development.
2 Related work of ChatGPT
In this section, we review the latest research related to the application, ethics,
and evaluation of ChatGPT.
2.1 Application of ChatGPT
2.1.1 Question And Answering
In the education ﬁeld
ChatGPT is commonly used for question and answers testing in the edu-
cation sector. Users can use ChatGPT to learn, compare and verify answers
for diﬀerent academic subjects such as physics, mathematics, and chemistry,
4

[chunk 1]
development.
2 Related work of ChatGPT
In this section, we review the latest research related to the application, ethics,
and evaluation of ChatGPT.
2.1 Application of ChatGPT
2.1.1 Question And Answering
In the education ﬁeld
ChatGPT is commonly used for question and answers testing in the edu-
cation sector. Users can use ChatGPT to learn, compare and verify answers
for diﬀerent academic subjects such as physics, mathematics, and chemistry,
4

[chunk 2]
development.
2 Related work of ChatGPT
In thi

In [21]:
vectorstore1 = create_vectorstore(chunk1,watsonx_enbedding,collection_name="gtp_research_watson1")
vectorstore2 = create_vectorstore(chunk2,ollama_enbedding,collection_name="gtp_research_watson2")

watson1_retriever = create_retriever(vectorstore1)
watson2_retriever = create_retriever(vectorstore2)

query = 'where ci use chatGPT?'

print_retrieved_docs('watsonx_enbedding',watson1_retriever, query)
print_retrieved_docs('ollama_enbedding',watson2_retriever, query)


watsonx_enbedding

[chunk 0]
development.
2 Related work of ChatGPT
In this section, we review the latest research related to the application, ethics,
and evaluation of ChatGPT.
2.1 Application of ChatGPT
2.1.1 Question And Answering
In the education ﬁeld
ChatGPT is commonly used for question and answers testing in the edu-
cation sector. Users can use ChatGPT to learn, compare and verify answers
for diﬀerent academic subjects such as physics, mathematics, and chemistry,
4

[chunk 1]
development.
2 Related work of ChatGPT
In this section, we review the latest research related to the application, ethics,
and evaluation of ChatGPT.
2.1 Application of ChatGPT
2.1.1 Question And Answering
In the education ﬁeld
ChatGPT is commonly used for question and answers testing in the edu-
cation sector. Users can use ChatGPT to learn, compare and verify answers
for diﬀerent academic subjects such as physics, mathematics, and chemistry,
4

[chunk 2]
development.
2 Related work of ChatGPT
In this sec

### 2. MMR(Maximal Marginal Relevace) Retriever
- 관련성(Relevace)과 다양성(Diversity) 고려
- 법률 문서, 기술 메뉴얼 처럼 유사 내용이 반복되는 문서에 효과적임
- 동작과정
    - research => embedding
    - vectorstore에서 research과 유사한 상위 fetch_k(후보 문서)를 추출
    - fetch_k 에서 MMR 점수 계산 => 가장 높은 문서 추출
    - 남은 fetch_에서 MMR 점수 계산 => 높은 문서 추출
    - 추출한 높은 문서에서 최종 k 반환

In [ ]:


chunk1 = create_chunk_from_pdf("./data/2026 상 삼성전자 DX부문 직무기술서.pdf",500,50)

vectorstore1 = create_vectorstore(chunk1,watsonx_enbedding,collection_name="samsung_watson1",persist_directory="./db/watson_chroma")

mmr_retriever = create_retriever(vectorstore1,search_type="mmr",k=5)
similarity_retriever = create_retriever(vectorstore1,k=5)

query = '마케팅 - 제품/서비스 마케팅 포지션은?'

print_retrieved_docs('MMR',mmr_retriever, query)
print_retrieved_docs('Similarity',similarity_retriever, query)


MMR

[chunk 0]
마케팅
마켓 센싱 및 정보 분석 결과를 바탕으로
 당사 제품과 서비스의 차별화 가치를 소비자에게 효과적인
커뮤니케이션 방법으로 전달하여 목표한 경영성과를 창출하고 브랜드 가치를 제고합니다

Page : 14

[chunk 1]
국내영업마케팅
국내의 각 분야별 영업 채널을 발굴
 지원하여 성과 창출과 지속 성장을 추구하는 동시에
 한국 시장에
대한 심도있는 분석을 통해 삼성전자
 부문 제품의 마케팅 전략을 수립 ⋅ 적용하고 글로벌
시장으로의 확산 기반을 마련합니다

Page : 24

[chunk 2]
해외영업
고객과 시장
 제품에 대한 이해를 바탕으로 시장 수요와 경쟁환경을 분석하여 국가
 거래선별 목표 설정
영업전략 수립
 신규 제품
 영업 채널을 발굴하고 판매전략 수립 및 실행을 통해 매출 극대화에
기여합니다

Page : 18

[chunk 3]
구매
제품 생산에 필요한 자원
 부품
 설비 및 제품
 을 최적의 품질과 가격으로
협상
 구매하고 시장의 수요 및 생산 계획에 맞춰 적기 공급하여 회사 경영에 기여합니다

Page : 26

[chunk 4]
품질/서비스
신제품 개발 신뢰성 검증
 공정 불량 검출
 고객 서비스 지원
 부품 협력사 관리 등 불량 없는 제품 생산
및 고객만족 실현을 위한 솔루션을 수립하여 제공합니다

Page : 12

Similarity

[chunk 0]
마케팅
마켓 센싱 및 정보 분석 결과를 바탕으로
 당사 제품과 서비스의 차별화 가치를 소비자에게 효과적인
커뮤니케이션 방법으로 전달하여 목표한 경영성과를 창출하고 브랜드 가치를 제고합니다

Page : 14

[chunk 1]
마케팅
마켓 센싱 및 정보 분석 결과를 바탕으로
 당사 제품과 서비스의 차별화 가치를 소비자에게 효과적인
커뮤니케이션 방법으로 전달하여 목표한 경영성과를 창출하고 브랜드 가치를 제고합니다

Page : 14

[chunk 2]
마케팅
마켓 센싱 및 정보 분석 결과를 바탕으로
 당사 제품과 서비스의 차별화 가치를 소비자에게 효과

### 3. SelfQuery Retriever
- 자연어 질문을 분석하여 시멘틱 검색 쿼리와 메타데이터필터를 LLM이자동으로 생성하게하는 고급 Retriever
- 질문 : 2023년 이후 계약금액이 1억 이상인 계약 찾아줘 => LLM
    - 시멘틱 검섹 쿼리 :계약
    - filtter:{year > = 2023, 계약금액 >= 100000 ~}

In [35]:
!pip3 install lark


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
docs = [
    Document(
        page_content="삼성전자 제품 마케팅 직무입니다.",
        metadata={
            "year" :2025,
            "department":"marketing"
        }
    ),
    Document(
        page_content="AI 연구 개발 직무입니다.",
        metadata={
            "year" :2024,
            "department":"ai"
        }
    ),
    Document(
        page_content="백엔드 개발 직무입니다.",
        metadata={
            "year" :2025,
            "department":"developer"
        }
    ),
]


metadata_feild_info = [
    AttributeInfo(name="year",description="문서 작성 연도", type="integer"),
    AttributeInfo(name="department",description="wlran qntj", type="string"),
]

document_content_description = "회사 내부 문서 및 직무 자료"

In [8]:
vectorstore1 = create_vectorstore(docs,watsonx_enbedding,collection_name="selfquery",persist_directory="./db/watson_chroma")

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=watson_llm,
    vectorstore=vectorstore1,
    document_contents=document_content_description,
    metadata_field_info=metadata_feild_info,
    verborse = True,
    enable_limit=True,
    structured_query_translator=ChromaTranslator()
)

In [9]:
question = "2024년 이후 ai 부서 직무 찾아줘"

self_query_retriever.invoke(question)

[Document(id='298655f7-2bf6-48ad-9eb8-ecdedf61caa4', metadata={'department': 'ai', 'year': 2024}, page_content='AI 연구 개발 직무입니다.')]

In [ ]:
docs = [
    Document(
        page_content="수분 가득한 히알루론산 세럼으로 피부 속 깊은 곳까지 수분을 공급합니다.",
        metadata={
            "year":2024,
            "category":"스킨케어",
            "user_rating":4
        }
    ),
    Document(
        page_content="24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.",
        metadata={
            "year":2023,
            "category":"메이크업",
            "user_rating":3
        }
    ),
    Document(
        page_content="식물성 성분으로 만든 저자극 클렌징 오일, 메이크업과 노폐물을 부드럽게 제거합니다.",
        metadata={
            "year":2023,
            "category":"클렌징",
            "user_rating":5
        }
    ),
    Document(
        page_content="비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.",
        metadata={
            "year":2023,
            "category":"스킨케어",
            "user_rating":2
        }
    ),
    Document(
        page_content="롱래스팅 립스틱, 선명한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.",
        metadata={
            "year":2024,
            "category":"메이크업",
            "user_rating":4
        }
    ),
    Document(
        page_content="자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA+++ 높은 자외선 차단 지수로 피부를 보호합니다.",
        metadata={
            "year":2025,
            "category":"선케어",
            "user_rating":5
        }
    ),
]

# 메타 데이터 필드 정보 생성
metadata_field_info = [
    AttributeInfo(
        name= "year",description="화장품 출시연도",type="integer"
    ),
    AttributeInfo(
        name= "category",description="화장품 카테고리['스킨케어','메이크업','클렌징','선케어']",type="string"
    ),
    AttributeInfo(
        name= "user_rating",description="화장품 평점(1~5)",type="integer"
    ),
]

document_content_description = "화장품 제품 정보"

vectorstore1 = create_vectorstore(docs,watsonx_enbedding,collection_name="selfquery",persist_directory="./db/watson_chroma")

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=watson_llm,
    vectorstore=vectorstore1,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose = True,
    enable_limit=True,
    structured_query_translator=ChromaTranslator()
)

In [9]:
self_query_retriever.invoke("2024년 이후로 평점이 4 이상인 제품을 추천해줘")

[Document(id='57dae01a-738f-49ea-9222-e691bd123b49', metadata={'year': 2024, 'category': '스킨케어', 'user_rating': 4}, page_content='수분 가득한'),
 Document(id='cb2a3217-0638-4ec8-b421-09f078e26ad8', metadata={'user_rating': 4, 'category': '스킨케어', 'year': 2024}, page_content='수분 가득한'),
 Document(id='ac3a33a9-ce42-4057-9e9a-780abe065e1b', metadata={'category': '메이크업', 'year': 2024, 'user_rating': 4}, page_content='롱래스틱 립스틱, 훌륭한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.'),
 Document(id='17e4e4a4-559e-4dc2-8813-464d35801bf3', metadata={'user_rating': 4, 'category': '메이크업', 'year': 2024}, page_content='롱래스틱 립스틱, 훌륭한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.')]

### OCR

In [11]:
pdf_path = "./data/2026 상 삼성전자 DX부문 직무기술서.pdf"
chunks = create_chunk_from_pdf(pdf_path)
for i in range(2):
    print("="*50)
    print(chunks[i].page_content[:500])

회로개발
회로 기술을 기반으로 삼성전자 제품 및 솔루션의 혁신적인 가치를 창출합니다
•
•
•
•
•
•
•
•
•
•
•
•
•
•


In [12]:
!pip3 install easyocr pymupdf

  Using cached pymupdf-1.27.2.3-cp310-abi3-win_amd64.whl.metadata (24 kB)
  Using cached torch-2.12.0-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached torchvision-0.27.0-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 33.4 MB/s eta 0:00:00
Using cached pymupdf-1.27.2.3-cp310-abi3-win_amd64.whl (19.2 MB)
Using cached torchvision-0.27.0-cp312-cp312-win_amd64.whl (4.0 MB)
Using cached torch-2.12.0-cp312-cp312-win_amd64.whl (123.0 MB)
   ---------------------------------------- 0.0/40.1 MB ? eta -:--:--
   ---- -----------------------------


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
# pdf => 이미지로 변환
import fitz

pdf = fitz.open(pdf_path)

for page_num in range(len(pdf)):
    page=pdf[page_num]
    pix = page.get_pixmap(dpi=300)
    pix.save(f"page_{page_num}.png")

In [15]:
# 이미지 => 텍스트
import easyocr

reader = easyocr.Reader(['ko','en'])
result = reader.readtext("page_19.png", detail=0, paragraph=True)
print("\n".join(result))

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.1% Complete

c:\souce\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


DA사업부(수원 근무) 해외영업 B2C 영업
포지션 소개 Job Overview
글로벌 시장과 소비자에 대한 이해틀 바탕으로 생활가전 제품의 판매 전락올 수립하고 매출 확대와 브랜드 리더십올 제고합니다:
수행업무 Job Details
담당 지역의 시장과 고객 특성, 판매 데이터들 분석하여 매출과 손의 극대화틀 위한 제품 가격유통 마켓팅올 아우르는 장단기 판매 전락올 수립합니다. 수립든 판매 전락올 기반으로 담당 법인과 현업하여 적기 공급올 위한 오퍼레이선올 지원하고 현지 법인의 실행 계획과 진척올 관리 및 개선합니다.
자격요건 Requirements
영어로 해외 자료 조사 및 커류니키이선이 가능하신 분 Global 이문화에 대한 이해도가 높으신 분 팀위크와 협업 능력올 보유하신 분
우대사항 Preferences
직무와 연관된 대내외 활동 경험올 보유하신 분 제2외국어 회화 역량울 보유하신 분
커리어 비전 Career Vision
담당 지역의 언어와 문화 습득올 포함한 글로벌 역량울 강화할 수 있으려 고객시장 유통 특성 등 전문 지식 습득과 법인 관리 및 개선올 통한 영업 실무 경험올 쌍울 수 있습니다. 삼성전자 해외영업 주재원으로서 현지 법인에서 근무하여, 매출 확대와 브랜드 리더십올 공고히 하는 현장 경험올 익히려 글로벌 영업 전문가로 성장할 수 있습니다:
'글로벌올 무대로 DA의 새로운 가능성울 열어갈 당신에게"
시장과 소비자에 대한 이해틀 바탕으로
생활가전 브랜드틀 최고의 자리로 이끌어 칼 인재틀 모십니다.


In [17]:
import json

pages =[]

for page_num in range(31):
    image_path = f"page_{page_num}.png"
    result = reader.readtext(image_path,detail=0,paragraph=True)
    text= "\n".join(result)
    pages.append({"page":page_num+1, "content":text})

with open('./data/samsung_dx_ocr.json',"w",encoding="utf-8") as f:
    json.dump(pages,f,ensure_ascii=False,indent=2)

c:\souce\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\souce\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\souce\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\souce\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\souce\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argume

### BM25
- 유사도 검색은 의미적 유사성 위주(제품명, 코드, 버전, 고유명사 등 키워드 마칭에 약함)
- 유사도 검색 단점 보완(semantic + sparse)
- 예
    - 환불 가능한 기간이 어떻게 되나요?(유사도 검색 유리)
    - ERR_CONNECTION 오류 => 키워드 검색

In [18]:
!pip3 install rank-bm25


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
with open('./data/samsung_dx_ocr.json',"r",encoding="utf-8") as f:
    pages = json.load(f)

docs = []

for page in pages:
    docs.append(
        Document(
            page_content=page['content'],
            metadata={
                "page":page['page'],
                "source":"samsung_dx_ocr"
            }
        )
    )

In [ ]:
# len(docs)

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

vectorstore = create_vectorstore(
    chunks=chunks,
    embeddings=watsonx_enbedding,
    persist_directory="./db/chroma_db",
    collection_name="samsung_dx"
)

In [24]:
# vectorstore.similarity_search("",k=3)

# 유사도 검색
dense_retriever = create_retriever(vectorstore)
# 키워드 검색 (법령, 제품명,...)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4
# weights : 검색 전략 0.4:0.6,dense:bm25
hybrid_retriver = EnsembleRetriever(retrievers=[dense_retriever,bm25_retriever], weights=[0.5,0.5])

query = "시스템 소프트웨어 자격요건에 운영체제 개념도 포함되어 있어?"

print_retrieved_docs("hybrid", hybrid_retriver,query)


hybrid

[chunk 0]
자격요건 Requirements
컴퓨터 전기 전자, 기계 로봇공학 등 관련 전공올 하신 분 운영체제 기본 개념에 대한 이해도틀 보유하신 분 요구사항울 분석하여 소프트웨어들 구조적으로 설계 및 구현하는 역량울 보유하신 분 다양한 분야의 엔지니어와 적극적으로 소통하여 협업할 수 짓는 역량울 보유하신 분
우대사항 Preferences
CIC++ 기반 시스템 프로그래망 역량울 보유하신 분 Linux 기반 임베디드 시스템 개발 경험올 보유하신 분(Kernel, Device Driver, BSP 등) CAN EtherCAT, UDP 등 하드웨어 통신 프로토록 활용 경험 보유하신 분 ROS2 기반 로보텍스 프로적트 수행 경험올 보유하신 분 Git 기반 협업 및 CICD 환경에서의 개발 경험올 보유하신 분
커리어 비전 Career Vision

Page : 7

[chunk 1]
미래로봇추진단(서울 근무) S/W개발 시스템 소프트웨어
포지션 소개 Job Overview
휴머노이드 로봇의 실시간 제어 시스템 및 소프트웨어 플렉품올 개발하는 직무입니다: 운영체제 환경 구성, 디바이스 드라이버, 실시간 제어 프레임위크 등 로봇 동작의 핵심 기반이 되는 소프트웨어름 설계 및 개발하다, 하드웨어 설계 조직 및 Al 연구 조직과 긴밀히 현업합니다:
수행업무 Job Details
휴머노이드 로봇의 실시간 제어 프레임위크 설계하고 개발합니다: 모터; 센서 등 로봇 하드웨어 제어름 위한 디바이스 드라이버 및 하드웨어 추상화 계층올 개발합니다. 로봇 제어용 운영체제(Linux 기반 실시간 OS 등) 환경올 구성하고 시스템 성능올 최적화합니다: 유관 부서와 협업하여 보행 조작 등 Al 기능과 제어 시스템 간 연동 미들웨어름 개발합니다:
자격요건 Requirements

Page : 7

[chunk 2]
S/W개발 소프트웨어 기술에 대한 전문적인 지식올 기반으로 창의적이고 분석적인 사고름 통해 신기술올 선도하고 당사 제품에 반영함으로써 제품 및 슬루선의 학신적

### ReRanking & Contextual Comprosstion
- 유사도 검색(Bi-Encoder) :쿼리(질문)와 문서를 각각 벡터로 변환 후 계산
- Cross-Encoder :쿼리+문서 쌍으로 벡터로 변환
    - 정확도가 매우 높으나, 매우 느림

In [25]:
!pip install cohere langchain-cohere sentence-transformers

  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   ---------------------------------------- 588.9/588.9 kB 10.2 MB/s  0:00:00
   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   ---------------------------------------  10.7/10.8 MB 61.0 MB/s eta 0:00:01
   ---------------------------------------- 10.8/10.8 MB 51.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 52.9 MB/s  0:00:00
Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl (341 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 8.0/8.0 MB 55.0 MB/s  0:00:00

   ---------- -----------------------------  3/12 [safetensors]
   ------------- --------------------------  4/12 [joblib]
   ------------- --------------------------  4/12 [joblib]
   ------------- -

In [ ]:
# 기본검색(=유사도 검색)
base_retriever = create_retriever(vectorstore,k=20)
query = "원격근무 정책은 어떻게 되나요?"
docs = base_retriever.invoke(query)

# 20개 후보군을 대상으로 Rernaking
reranker = CohereRerank(model="rerank-v4.0-pro", top_n=5)

compression_retriecer = ContextualCompressionRetriever(base_compressor=reranker,base_retriever=base_retriever)

# 최종
print_retrieved_docs("Rerank",compression_retriecer,query)


Rerank

[chunk 0]
한국종팔(전국 근무) 국내영업마켓팅
포지션 소개 Job Overview
삼성전자 DX부문에서 국내 시장에 출시하는 전 제품과 서비스의 국내 영업 마켓팅올 총팔하는 직무입니다. 온 오프라인 유통 채널별 영업 전락 및 제품과 브랜드의 마켓팅 전락올 수립하다 경영 성과름 창출하고 있습니다.
B2CB2B온라인 경로의 판매 전락올 수립하고 신규 거래선과 Biz 틀 발출합니다: 제품의 품목별 매출 손의올 관리하고 소비자 분석올 통한 가격 라인업 프로모선 등 효과적인 마켓팅 전락올 수립합니다: 국내시장 판매 목표름 설정하고 공급망올 관리(SCM)하여 담당 품목의 프트플리오 전반올 총팔합니다. 소비자에게 제품 및 브랜드 가치 전달하기 위한 광고캠페인 등 (MC전락올 수립합니다. 오프라인 매장의 리타일 마켓팅 전락올 수립합니다. 판매 직원 교육 관리 현장 Data 분석 등 소비자 접점의 판매 경쟁력올 높입니다.
수행업무 Job Details
자격요건 Requirements

Page : 26

[chunk 1]
우대사항 Preferences
모바일 기기 개발의 핵심 엔지니어로 성장, 차세대 스마트혼 웨어러블 등 주력 제품 개발에 직접 참여하고 미래 IT 분야의 리더로 도약할 수 있습니다. 입사 후, 사내 기술연구 특히 출원, 신제품 개발 경험올 쌓으면서 회로설계 전문가을 넘어 시스템 설계 품질 등 폭넓은 분야로 커리어 확장이 가능합니다 또한 Al, 시물레이선 등 소프트웨어와 하드웨어 움합올 통해 시장울 선도하는 인재로 성장할 수 있습니다:
커리어 비전 Career Vision
"가장 강력한 하드웨어 학신 당신과 Galaxy가 만나면 가능합나다'
창의적인 아이디어와 열정올 가진 인재틀 찾고 있습니다.
도전적이고 빠르게 변화하는 환경에서 함께 성장하고 싶다면 주저하지 말고 지원해 주세요

Page : 3

[chunk 2]
DA사업부(수원 근무) 마켓팅 제품) 서비스 마게팅
포지션 소개 Job Overview
고객 시장 경쟁사 기술 트랜드에 대한

### LLMChainExtractor
- 질문+문서를 같이 LLM에게 보내서 질문과 관련된 내용만 추출
- 현재 chunk 안에 있는 내용을 줄여내는 것

In [ ]:
# 문서 load / chunk 추출
chunks = create_chunk_from_pdf("./data/제주관광가이드.pdf")
# 인덱싱 - vectorstore
vectorstore = create_vectorstore(chunks,watsonx_enbedding,collection_name="jeju_guid")
#질의
base_retriever = create_retriever(vectorstore=vectorstore,k=20)

docs = base_retriever.invoke("생활 속 제주어에서 엄불랑허다는 무슨 뜻이야?")
for doc in docs:
    print(doc.page_content[:500])
    print('-'*10)

생활 속 제주어
제주는 타 지역보다 한국어의 고형(古形)을 많이 유지하고 있는 동시에 
제주도만의 고유한 어휘나 문법적 특성을 가지고 있다.
다른 지역 사람이 못 알아듣는 제주어
제주어 뜻풀이
솔쩨기 살짝
안네다 드리다
베지근허다 입안에 기름기가 감돌아 맛이 있다.
엄불랑허다 어마어마하다
코시롱허다 고소하다
산도록허다 시원하다  예) 물이 산도록헌 게 좋다.
두령청이 우두망찰
무사 왜
영, 경, 정 이렇게, 그렇게, 저렇게
게메 글쎄
인사말
제주어 뜻풀이
펜안허우꽈? 편안(안녕)하십니까?
제주도 오난 어떵허우꽈? 제주도에 오니 어떠십니까?
차말로 좋수다. 참말로 좋습니다.
공기도 마고, 산이영 바다잉여 마딱 좋은게마씀 공기도 맑고, 산이랑 바다랑 모두 좋네요.
서울 갈 때랑 하영 다앙 갑서. 서울 갈 때는 많이 담아서 가십시오.
게메양. 경 헤시민 얼마나 좋코마씀? 글쎄요. 그렇게 했으면 얼마나 좋겠습니까?
식당에서
제주어 뜻풀이
무신걸 먹으코? 무엇을 먹을까?
----------
생활 속 제주어
제주는 타 지역보다 한국어의 고형(古形)을 많이 유지하고 있는 동시에 
제주도만의 고유한 어휘나 문법적 특성을 가지고 있다.
다른 지역 사람이 못 알아듣는 제주어
제주어 뜻풀이
솔쩨기 살짝
안네다 드리다
베지근허다 입안에 기름기가 감돌아 맛이 있다.
엄불랑허다 어마어마하다
코시롱허다 고소하다
산도록허다 시원하다  예) 물이 산도록헌 게 좋다.
두령청이 우두망찰
무사 왜
영, 경, 정 이렇게, 그렇게, 저렇게
게메 글쎄
인사말
제주어 뜻풀이
펜안허우꽈? 편안(안녕)하십니까?
제주도 오난 어떵허우꽈? 제주도에 오니 어떠십니까?
차말로 좋수다. 참말로 좋습니다.
공기도 마고, 산이영 바다잉여 마딱 좋은게마씀 공기도 맑고, 산이랑 바다랑 모두 좋네요.
서울 갈 때랑 하영 다앙 갑서. 서울 갈 때는 많이 담아서 가십시오.
게메양. 경 헤시민 얼마나 좋코마씀? 글쎄요. 그렇게 했으면 얼마나 좋겠습니까?
식당에서
제주어 뜻풀이
무신걸 먹으코? 무엇을 먹을까?
----------


In [ ]:
# 답변만 축약

extractor = LLMChainExtractor.from_llm(watson_llm)

compression_retriever = ContextualCompressionRetriever(base_compressor=extractor, base_retriever=base_retriever)

docs = compression_retriever.invoke("생활 속 제주어에서 엄불랑허다는 무슨 뜻이야?")
for doc in docs:
    print(doc.page_content[:500])
    print('-'*10)

엄불랑허다 어마어마하다
----------
엄불랑허다 어마어마하다
----------
엄불랑허다는 무슨 뜻이야?
----------
엄불랑허다는 무슨 뜻이야?
----------


### Embedding Filter
- 임계값 기준으로 미달한 문서 제외

In [42]:
embedding_filter = EmbeddingsFilter(embeddings=watsonx_enbedding, similarity_threshold=0.8)

pipeline= DocumentCompressorPipeline(transformers=[embedding_filter,extractor])

compression_retriever = ContextualCompressionRetriever(base_compressor=pipeline, base_retriever=base_retriever)

docs = compression_retriever.invoke("생활 속 제주어에서 엄불랑허다는 무슨 뜻이야?")
for doc in docs:
    print(doc.page_content[:500])
    print('-'*10)

엄불랑허다 어마어마하다
----------
엄불랑허다 어마어마하다
----------
엄불랑허다는 무슨 뜻이야?
----------
엄불랑허다는 무슨 뜻이야?
----------


In [43]:
filterd_docs = embedding_filter.compress_documents(docs,query)
len(filterd_docs)

0

#### RAG 성능개선
1. 문서 전처리
- chunk 최적화, metadata 추가

2. Retrievl(검색) 개선
- MMR, BM25, Hybrid, SelfQuery

3. Retrievl 후 처리
- Rerank, EmbeddingFilter, LLMChainExtractor, ContextualCompressionRetriever

In [47]:
# 한양대 대학원 캠퍼스 가이드 pdf 로드 후 chunk_size = 500, overlap=50 split 한 후 사이즈 확인
# 청크 내용 확인
pdf_path = "./data/한양대 대학원 캠퍼스 가이드.pdf"
chunks = create_chunk_from_pdf(pdf_path)
len(chunks)

# pdf->이미지->텍스트 인식 json
pdf = fitz.open(pdf_path)

for page_num in range(len(pdf)):
    page=pdf[page_num]
    pix = page.get_pixmap(dpi=300)
    pix.save(f"page_{page_num}.png")

reader = easyocr.Reader(['ko','en'])

result = reader.readtext("page_19.png", detail=0, paragraph=True)
print("\n".join(result))

pages =[]

for page_num in range(len(pdf)):
    image_path = f"page_{page_num}.png"
    result = reader.readtext(image_path,detail=0,paragraph=True)
    text= "\n".join(result)
    pages.append({"page":page_num+1, "content":text})

with open('./data/hanyang_campus.json',"w",encoding="utf-8") as f:
    json.dump(pages,f,ensure_ascii=False,indent=2)

# 벡터 스토어에 저장 collection_name="hanyang_campus"
with open('./data/hanyang_campus.json',"r",encoding="utf-8") as f:
    pages = json.load(f)

docs = []

for page in pages:
    docs.append(
        Document(
            page_content=page['content'],
            metadata={
                "page":page['page'],
                "source":"hanyang_campus"
            }
        )
    )

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

vectorstore = create_vectorstore(
    chunks=chunks,
    embeddings=watsonx_enbedding,
    persist_directory="./db/chroma_db",
    collection_name="hanyang_campus"
)

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
c:\souce\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


I. 개요
(5) 운영 회장 이하 12명의 간부로 학생회틀 이끌어 가고 있습니다 총 예산은 매 학기 원우들이 내는 원우회비와 각종 행사률 위한 학교 지원금으로 구성되어 있으려 원우들올 위한 다양한 복지 및 행사에 집행되고 있습니다: 참고로 원우회비틀 납부하신 분에 한해서만 진행되는 행사들도 있으니 원우회비률 납부하서서 더 많은 혜택올 받아 가시기 바람니다
(6) 활동 학생회에서 매년 정기적으로 진행하는 활동은 보통 아래와 같으  이밖에도 도서 지원 사업 졸업 기념 이번트 등 원우들올 위해 여러 가지 활동올 하고 있습니다
I. 학사안내
1) 춘 추계 원우 한마당 개최 춘계(4 5월) 추계(9 10월) 원우한마당올 매년 개최하고 있으려 이 자리는 학과루 떠나 원우들 간의 친목과 화합울 도모하는 장입니다: 많은 상품과 교우 관계 네트위크 구축을 통해서 즐거운 대학원 생활올 누리실 수 있습니다: 부가적으로 따르는 상품 및 기념품도 꼭 확인하시기 바람니다:
1 로드맵으로 본 대학원 생활 2 주요 학사안내 3. 기타 학사안내 4. 학생 종합정보 프로그램 안내
2) 여름방학 해외 단기 연수 프로그램 여름방학기간올 활용하여 매년 정기적으로 이루어지고 있습니다: 해외 대학 팀방 등올 포함한 해외 단기 연수름 통해 원우들이 보다 넓은 세계로 나가기 위한 견문을 넓히고 새롭고 다양한 경험올 통하여 원우들이 한 단계 더 나아갈 수 잇는 밀거름이 되길 바람니다:
매해컨흉대 대학원 인우가 되는 신입생들어계 이률 기념하여 정성이 담긴 작은기념품올 제작하여 배부하고 있습니다: 또한 원우 한마당 기념품도 제작하여 배부하고 있습니다
ID
4) 특별강좌 개설 원우들의 논문작성이나 수업에 관련하여 현실적인 도움이 월 수 있도록 외부에서 전문 강사흘 초방하여 특별강좌흘 개설하고 있습니다: 현재는 방학 기간을 이용하여 SPSS STATA(통계 프로그램)과 같은 통계강좌 및 논문 작성 방법 등 원우들에게 유의할 만한 특강울 개설하고 있으려 앞으로 설문과 홍보름 통해 많은 원우들이 필요로 하는 강좌들 

c:\souce\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\souce\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\souce\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\souce\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\souce\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argume

KeyboardInterrupt: 

In [ ]:
hanyang_vectorstore = create_vectorstore(
    chunks=chunks,
    embeddings=watsonx_enbedding,
    persist_directory="./db/chroma_db",
    collection_name="hanyang_campus"
)